# 03c — Deep Learning Baselines

**RAKSHAK-ICS** | Week 3 — Anomaly Transformer + USAD

Journal-grade DL baselines for comparison with our LSTM-AE + GAT fusion:

| Model | Paper | Venue | Year |
|-------|-------|-------|------|
| Anomaly Transformer | Time Series Anomaly Detection with Association Discrepancy | ICLR | 2022 |
| USAD | UnSupervised Anomaly Detection on Multivariate Time Series | KDD | 2020 |

Both are implemented as lightweight versions suitable for SWaT A9 (65 features, 60-step windows).

In [ ]:
import os
os.chdir('..')

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

plt.style.use('dark_background')
PALETTE = ['#00d4ff', '#ff6b6b', '#ffd93d', '#6bcb77', '#c084fc', '#ff922b']
sns.set_palette(PALETTE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Setup complete.')

## 1. Load Data

In [ ]:
from src.preprocess import load_processed_data
from src.baselines import inject_synthetic_anomalies, compute_metrics

data = load_processed_data('data/proof/')
X_train = data['X_train']  # (N, 60, 65)
X_val = data['X_val']
X_test = data['X_test']

print(f'Train: {X_train.shape}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')

## 2. USAD (KDD 2020)

**UnSupervised Anomaly Detection** uses dual autoencoders with adversarial training:
- Encoder shared between AE1 and AE2
- AE1 reconstructs input; AE2 reconstructs AE1's output
- Anomaly score = α * |X - AE1(X)| + (1-α) * |X - AE2(AE1(X))|

In [ ]:
class USADEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, latent_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
    def forward(self, x):
        return self.net(x)

class USADDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim=32, output_dim=65):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, output_dim),
            nn.Sigmoid(),
        )
    def forward(self, z):
        return self.net(z)

class USAD(nn.Module):
    def __init__(self, window_size=60, n_features=65, hidden_dim=32, latent_dim=16):
        super().__init__()
        input_dim = window_size * n_features
        self.encoder = USADEncoder(input_dim, hidden_dim, latent_dim)
        self.decoder1 = USADDecoder(latent_dim, hidden_dim, input_dim)
        self.decoder2 = USADDecoder(latent_dim, hidden_dim, input_dim)
        self.window_size = window_size
        self.n_features = n_features
    
    def forward(self, x):
        # x: (batch, window, features)
        x_flat = x.reshape(x.size(0), -1)
        z = self.encoder(x_flat)
        w1 = self.decoder1(z)  # AE1 reconstruction
        w2 = self.decoder2(z)  # AE2 reconstruction
        w3 = self.decoder2(self.encoder(w1))  # AE2(AE1(x))
        return w1, w2, w3, x_flat
    
    def anomaly_score(self, x, alpha=0.5):
        with torch.no_grad():
            w1, w2, w3, x_flat = self.forward(x)
            score1 = torch.mean((x_flat - w1) ** 2, dim=1)
            score2 = torch.mean((x_flat - w3) ** 2, dim=1)
            return alpha * score1 + (1 - alpha) * score2

print('USAD model defined.')

In [ ]:
# Train USAD
EPOCHS = 30
BATCH_SIZE = 256
LR = 1e-3

usad = USAD(window_size=60, n_features=65).to(device)
optimizer = torch.optim.Adam(usad.parameters(), lr=LR)

train_tensor = torch.FloatTensor(X_train).to(device)
train_loader = DataLoader(TensorDataset(train_tensor), batch_size=BATCH_SIZE, shuffle=True)

losses = []
t0 = time.time()

for epoch in range(EPOCHS):
    usad.train()
    epoch_loss = 0
    n_batches = 0
    
    for (batch,) in train_loader:
        w1, w2, w3, x_flat = usad(batch)
        
        # USAD loss: L = (1/n+1)*L_AE1 + (n/n+1)*L_AE2
        n = epoch + 1
        loss_ae1 = (1.0 / (n + 1)) * torch.mean((x_flat - w1) ** 2)
        loss_ae2 = (n / (n + 1)) * torch.mean((x_flat - w3) ** 2)
        loss = loss_ae1 + loss_ae2
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.6f}')

train_time_usad = time.time() - t0
print(f'\nUSAD training complete in {train_time_usad:.1f}s')

In [ ]:
# USAD training loss curve
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, EPOCHS+1), losses, 'o-', color=PALETTE[0], linewidth=2, markersize=4)
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Loss', fontsize=13)
ax.set_title('USAD Training Loss', fontsize=16, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/usad_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Anomaly Transformer (ICLR 2022) — Simplified

Simplified version focusing on the **Association Discrepancy** concept:
- Uses self-attention to model temporal associations
- Anomaly score based on attention distribution divergence

In [ ]:
class AnomalyAttention(nn.Module):
    """Simplified anomaly attention with prior/series association."""
    def __init__(self, d_model, n_heads=4, d_ff=128, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
    
    def forward(self, x):
        attn_out, attn_weights = self.attention(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x, attn_weights

class AnomalyTransformer(nn.Module):
    def __init__(self, n_features=65, d_model=64, n_heads=4, n_layers=2, d_ff=128, window_size=60):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, window_size, d_model) * 0.02)
        self.layers = nn.ModuleList([AnomalyAttention(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.output_proj = nn.Linear(d_model, n_features)
        self.window_size = window_size
        self.n_features = n_features
    
    def forward(self, x):
        # x: (batch, window, features)
        h = self.input_proj(x) + self.pos_embed
        all_attn = []
        for layer in self.layers:
            h, attn = layer(h)
            all_attn.append(attn)
        out = self.output_proj(h)
        return out, all_attn
    
    def anomaly_score(self, x):
        with torch.no_grad():
            recon, attn_list = self.forward(x)
            # Reconstruction error
            recon_err = torch.mean((x - recon) ** 2, dim=(1, 2))
            # Association discrepancy: KL div of attention from uniform
            T = x.size(1)
            uniform = torch.ones(T, T, device=x.device) / T
            attn_mean = torch.stack(attn_list).mean(0).mean(1)  # avg over layers and heads
            kl_div = torch.sum(attn_mean * torch.log(attn_mean / uniform + 1e-8), dim=-1).mean(dim=-1)
            return recon_err + 0.5 * kl_div

print('Anomaly Transformer model defined.')

In [ ]:
# Train Anomaly Transformer
AT_EPOCHS = 30

at_model = AnomalyTransformer(n_features=65, d_model=64, n_heads=4, n_layers=2).to(device)
at_optimizer = torch.optim.Adam(at_model.parameters(), lr=1e-3)

at_losses = []
t0 = time.time()

for epoch in range(AT_EPOCHS):
    at_model.train()
    epoch_loss = 0
    n_batches = 0
    
    for (batch,) in train_loader:
        recon, attn_list = at_model(batch)
        loss = torch.mean((batch - recon) ** 2)
        
        at_optimizer.zero_grad()
        loss.backward()
        at_optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg = epoch_loss / n_batches
    at_losses.append(avg)
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{AT_EPOCHS}, Loss: {avg:.6f}')

train_time_at = time.time() - t0
print(f'\nAnomaly Transformer training complete in {train_time_at:.1f}s')

## 4. Evaluate Both DL Baselines

In [ ]:
from src.baselines import flatten_windows

SEEDS = [42, 123, 456, 789, 1024]
dl_results = {}

for model_name, model_obj, score_fn in [
    ('USAD', usad, lambda m, x: m.anomaly_score(x).cpu().numpy()),
    ('Anomaly Transformer', at_model, lambda m, x: m.anomaly_score(x).cpu().numpy()),
]:
    print(f'\nEvaluating {model_name}...')
    model_obj.eval()
    seed_metrics = []
    
    for seed in SEEDS:
        # Get anomaly scores on test set
        test_tensor = torch.FloatTensor(X_test).to(device)
        scores = []
        
        with torch.no_grad():
            for i in range(0, len(test_tensor), BATCH_SIZE):
                batch = test_tensor[i:i+BATCH_SIZE]
                s = score_fn(model_obj, batch)
                scores.append(s)
        
        scores = np.concatenate(scores)
        
        # Generate labels and threshold
        _, y_test = inject_synthetic_anomalies(
            flatten_windows(X_test), 0.05, seed=seed + 1000
        )
        threshold = np.percentile(scores, 95)
        y_pred = (scores > threshold).astype(int)
        
        metrics = compute_metrics(y_test, y_pred, scores)
        seed_metrics.append(metrics)
        print(f'  Seed {seed}: F1={metrics["f1"]:.4f}, AUC={metrics["auc_roc"]:.4f}')
    
    agg = {}
    for metric in ['f1', 'precision', 'recall', 'auc_roc']:
        vals = [m[metric] for m in seed_metrics]
        agg[metric] = {
            'mean': float(np.mean(vals)),
            'std': float(np.std(vals)),
            'formatted': f'{np.mean(vals):.4f}\u00b1{np.std(vals):.4f}',
        }
    dl_results[model_name] = {'aggregated': agg}
    print(f'  >> {model_name}: F1={agg["f1"]["formatted"]}')

## 5. DL Baselines Comparison

In [ ]:
dl_names = list(dl_results.keys())
metrics_to_plot = ['f1', 'precision', 'recall', 'auc_roc']

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(18, 5))

for i, metric in enumerate(metrics_to_plot):
    means = [dl_results[n]['aggregated'][metric]['mean'] for n in dl_names]
    stds = [dl_results[n]['aggregated'][metric]['std'] for n in dl_names]
    
    axes[i].bar(range(len(dl_names)), means, yerr=stds, capsize=5,
                color=[PALETTE[0], PALETTE[1]], edgecolor='white', linewidth=0.5)
    axes[i].set_xticks(range(len(dl_names)))
    axes[i].set_xticklabels(dl_names, fontsize=10)
    axes[i].set_title(metric.replace('_', ' ').upper(), fontsize=14, fontweight='bold')
    axes[i].set_ylim(0, 1.05)
    axes[i].grid(axis='y', alpha=0.3)
    for j, (m, s) in enumerate(zip(means, stds)):
        axes[i].text(j, m + s + 0.02, f'{m:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('DL Baseline Comparison (mean \u00b1 std, 5 seeds)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/dl_baselines_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Training Loss Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, len(losses)+1), losses, 'o-', color=PALETTE[0], linewidth=2, markersize=4, label='USAD')
ax.plot(range(1, len(at_losses)+1), at_losses, 's-', color=PALETTE[1], linewidth=2, markersize=4, label='Anomaly Transformer')
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Loss', fontsize=13)
ax.set_title('DL Baseline Training Loss Curves', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/dl_baselines_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Results

In [ ]:
import json

os.makedirs('results/tables', exist_ok=True)
with open('results/tables/dl_baseline_results.json', 'w') as f:
    json.dump(dl_results, f, indent=2, default=str)

# Save models
os.makedirs('models/baselines', exist_ok=True)
torch.save(usad.state_dict(), 'models/baselines/usad.pt')
torch.save(at_model.state_dict(), 'models/baselines/anomaly_transformer.pt')

print('Saved: results/tables/dl_baseline_results.json')
print('Saved: models/baselines/usad.pt')
print('Saved: models/baselines/anomaly_transformer.pt')

## 8. Key Findings

### DL Baselines Summary
- **USAD** uses dual autoencoders with adversarial training
- **Anomaly Transformer** uses self-attention association discrepancy
- Both trained reconstruction-based on normal SWaT A9 data
- These serve as strong DL baselines for journal comparison

### Next Steps
- **Week 4**: LSTM Autoencoder (Blue Agent Stream 1)
- **Week 5**: GAT GNN (Blue Agent Stream 2) + Fusion
- Our LSTM-AE + GAT should outperform both USAD and Anomaly Transformer